<a href="https://colab.research.google.com/github/arturocarvo/Proyecto-Final-Data-Science/blob/main/Proyecto_Final_NLP_vs_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de Inteligencia Logística en E-commerce

# Introducción
El presente proyecto final aplica técnicas de Procesamiento de Lenguaje Natural (NLP) y Deep Learning sobre un dataset real de una operación logística en CABA. El objetivo es doble: por un lado, normalizar las instrucciones de entrega de los clientes y, por otro, predecir el valor estratégico de los pedidos mediante una arquitectura de red neuronal profunda.

# Procesamiento de Lenguaje Natural
Para esta sección, utilicé la columna Notas del comprador, la cual contiene texto no estructurado con indicaciones de envío. Se desarrollaron las siguientes tareas de preprocesamiento:

Limpieza y Normalización: Conversión a minúsculas y eliminación de caracteres especiales, números y símbolos mediante expresiones regulares (Regex).

Tokenización: Fragmentación del texto en unidades léxicas (palabras) utilizando la librería NLTK.

Eliminación de Stopwords: Filtrado de palabras vacías del idioma español (como "el", "la", "de") que no aportan valor semántico al análisis logístico.

In [10]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1CvWjwE8nGAdWmC_UheThCV_fdpuTQC_QoUjHSNYZxLY/export?format=csv"

df = pd.read_csv(url)

columnas_utiles = [
    'Subtotal de productos', 'Descuento', 'Costo de envío', 'Total',
    'Provincia o estado', 'Localidad', 'Medio de envío', 'Medio de pago',
    'Fecha', 'Estado del envío', 'Notas del comprador'
]

df_proyecto = df[columnas_utiles].copy()


df_proyecto['Notas del comprador'] = df_proyecto['Notas del comprador'].fillna('Sin nota')


print("Conexión exitosa")
print(df_proyecto.head())

Conexión exitosa
   Subtotal de productos  Descuento  Costo de envío    Total  \
0                53800.0     5380.0          6500.0  54920.0   
1                52400.0     5240.0          6500.0  53660.0   
2                62400.0        0.0          6500.0  68900.0   
3                63800.0        0.0          6500.0  70300.0   
4                52400.0        0.0          6500.0  58900.0   

  Provincia o estado                        Localidad Medio de envío  \
0    Capital Federal  Ciudad Autonoma de Buenos Aires      MOTO CABA   
1    Capital Federal                              NaN      MOTO CABA   
2    Capital Federal                         Recoleta      MOTO CABA   
3    Capital Federal                        Mataderos      MOTO CABA   
4    Capital Federal                          Palermo      MOTO CABA   

  Medio de pago                Fecha Estado del envío  \
0     Pago Nube   17/03/2026 6:26:37          Enviado   
1     Pago Nube   17/03/2026 0:34:47          Envia

In [11]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

palabras_irrelevantes = set(stopwords.words('spanish'))

def preprocesar_texto(texto):
    texto = str(texto).lower()

    texto = re.sub(r'[^a-zñáéíóú]', ' ', texto)

    tokens = word_tokenize(texto)

    tokens_limpios = [w for w in tokens if w not in palabras_irrelevantes and len(w) > 2]

    return tokens_limpios

df_proyecto['Notas_Procesadas'] = df_proyecto['Notas del comprador'].apply(preprocesar_texto)


print(f"Original: {df_proyecto['Notas del comprador'].iloc[0]}")
print(f"Procesado: {df_proyecto['Notas_Procesadas'].iloc[0]}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Original: Sin nota
Procesado: ['nota']


In [12]:
import numpy as np

promedio_venta = df_proyecto['Total'].mean()
print(f"El ticket promedio de tu tienda es: ${promedio_venta:.2f}")

df_proyecto['Clase_Ticket'] = (df_proyecto['Total'] > promedio_venta).astype(int)

df_proyecto['Fecha'] = pd.to_datetime(df_proyecto['Fecha'], errors='coerce')
df_proyecto['Dia_Semana'] = df_proyecto['Fecha'].dt.dayofweek.fillna(0)

print("Variable 'Clase_Ticket' creada")

El ticket promedio de tu tienda es: $68766.52
Variable 'Clase_Ticket' creada


/tmp/ipykernel_8964/4229027726.py:8: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_proyecto['Fecha'] = pd.to_datetime(df_proyecto['Fecha'], errors='coerce')


# Deep Learning

Construcción de una primera red neuronal sencilla.

Se diseñó un modelo de clasificación binaria para segmentar los pedidos. El objetivo inicial fue predecir la categoría del ticket (Alto/Bajo) basándose en variables numéricas.

Variables de entrada (Features): Subtotal de productos y Día de la semana.

Arquitectura: Una red neuronal tipo Perceptrón con una capa de entrada y una capa de salida utilizando la función de activación Sigmoid para obtener una probabilidad binaria.

In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_proyecto[['Dia_Semana', 'Subtotal de productos']].fillna(0)
y = df_proyecto['Clase_Ticket']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

modelo_ticket = Sequential([
    Dense(12, activation='relu', input_shape=(X_train.shape[1],)), # Capa oculta
    Dense(1, activation='sigmoid')                                 # Salida binaria (Alto/Bajo)
])

modelo_ticket.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Entrenando la red para predecir categoría de ticket...")
modelo_ticket.fit(X_train, y_train, epochs=15, validation_split=0.2, verbose=1)

Entrenando la red para predecir categoría de ticket...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


182/182 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6885 - loss: 0.5896 - val_accuracy: 0.9310 - val_loss: 0.4819
Epoch 2/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9489 - loss: 0.3801 - val_accuracy: 0.9648 - val_loss: 0.2804
Epoch 3/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9608 - loss: 0.2188 - val_accuracy: 0.9628 - val_loss: 0.1773
Epoch 4/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9646 - loss: 0.1508 - val_accuracy: 0.9634 - val_loss: 0.1352
Epoch 5/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9636 - loss: 0.1199 - val_accuracy: 0.9641 - val_loss: 0.1120
Epoch 6/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9671 - loss: 0.1016 - val_accuracy: 0.9669 - val_loss: 0.0972
Epoch 7/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9672 - loss: 0.0902 - val_accuracy: 0.9683 - val_loss: 0.0876
Epoch 8/15
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9681 - loss: 0.0824 - val_accuracy: 0.9683 - val_

# Mejora de la Red Neuronal

Profundizar en Deep Learning mediante la adición de capas y optimización del modelo.

Para el trabajo final, opté por transformar la red inicial en una Red Neuronal Profunda (Deep Neural Network). Las mejoras implementadas para optimizar el rendimiento fueron:

Aumento de Profundidad: Se agregaron tres capas densas ocultas (con 32, 16 y 8 neuronas respectivamente) para capturar relaciones no lineales complejas entre el monto de la venta y el momento de la compra.

Funciones de Activación: Se utilizó ReLU (Rectified Linear Unit) en las capas ocultas para evitar el desvanecimiento del gradiente y mejorar la velocidad de aprendizaje.

Regularización con Dropout: Se incluyó una capa de Dropout (0.2), una técnica profesional que desactiva aleatoriamente el 20% de las neuronas durante el entrenamiento para prevenir el overfitting (sobreajuste) y garantizar que el modelo generalice bien ante datos nuevos.

Optimización: Se configuró el optimizador Adam y se extendió el entrenamiento a 30 épocas.

In [15]:
from tensorflow.keras.layers import Dropout

modelo_final = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)), # Capa 1: 32 neuronas
    Dropout(0.2),                                                  # Técnica profesional de regularización
    Dense(16, activation='relu'),                                  # Capa 2: 16 neuronas (Profundidad)
    Dense(8, activation='relu'),                                   # Capa 3: 8 neuronas
    Dense(1, activation='sigmoid')                                 # Salida
])

modelo_final.compile(optimizer='adam',
                     loss='binary_crossentropy',
                     metrics=['accuracy'])

print("Entrenando")
historia_final = modelo_final.fit(X_train, y_train,
                                  epochs=30,
                                  validation_split=0.2,
                                  verbose=1)

print("Red Neuronal Profunda entrenada con éxito")

Entrenando
Epoch 1/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9182 - loss: 0.3328 - val_accuracy: 0.9697 - val_loss: 0.1062
Epoch 2/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9650 - loss: 0.0872 - val_accuracy: 0.9655 - val_loss: 0.0638
Epoch 3/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9672 - loss: 0.0691 - val_accuracy: 0.9752 - val_loss: 0.0513
Epoch 4/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9707 - loss: 0.0629 - val_accuracy: 0.9848 - val_loss: 0.0483
Epoch 5/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9702 - loss: 0.0606 - val_accuracy: 0.9814 - val_loss: 0.0464
Epoch 6/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9734 - loss: 0.0545 - val_accuracy: 0.9724 - val_loss: 0.0452
Epoch 7/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9746 - loss: 0.0532 - val_accuracy: 0.9717 - val_loss: 0.0453
Epoch 8/30
182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9748 - loss: 0.0520 - val_a

# Conclusión y Análisis de Resultados
El modelo final alcanzó una precisión (Accuracy) superior al 97% en el set de validación. Esto demuestra que es posible automatizar la categorización de clientes basándose en su comportamiento histórico de compra.

Desde una perspectiva de negocio, esta herramienta permite a la empresa de logística:

Priorizar envíos de "Ticket Alto".

Asignar recursos de manera eficiente según el día de la semana.

Automatizar la gestión de datos que provienen de plataformas como Tienda Nube sin intervención manual constante.